# Parcial 2 - Inteligencia Artificial

**Parte 2: laboratorio (50 puntos)**  
**Tiempo previsto:** 50 minutos. Con recomendación vigente de tiempo adicional: hasta 70 minutos, salvo indicación individual diferente.  
**Modalidad:** individual, con materiales e internet, sin IA generativa.

Todas las personas reciben el mismo cuaderno, las mismas consignas y el mismo puntaje.

Los dos ejercicios son variaciones de trabajos realizados en el curso: representación de texto, embeddings, búsqueda semántica, LLMs, prompting y RAG.

El código largo ya está preparado y aparece plegado. No necesitás leerlo para resolver. En cada ejercicio hay una celda para **ejecutar sin cambios** y una celda corta `TU TRABAJO` para elegir alternativas. Al terminar, elegí `Entorno de ejecución (Runtime) > Ejecutar todo (Run all)`, guardá las salidas y descargá el archivo `.ipynb`.

**Antes de empezar:** activá la GPU (*Entorno de ejecución* > *Cambiar tipo de entorno de ejecución* > *T4 GPU*) y ejecutá la celda **Preparación del entorno**. Descarga tres modelos (alrededor de 4,5 GB en total) y tarda dos o tres minutos. Mientras tanto, leé las consignas.

## Qué tenés que lograr

Vas a trabajar sobre un asistente para la **Mesa de Ayuda de la Universidad Católica del Uruguay (UCU)**: estudiantes que preguntan por contraseñas, Wi-Fi, impresiones, biblioteca y laboratorios. Las respuestas están en 16 fragmentos de documentación interna.

- **Ejercicio 1:** comparar formas de **buscar** el fragmento correcto para cada consulta.
- **Ejercicio 2:** comparar formas de **responder** con un LLM chico, con y sin RAG.
- **Cierre:** responder cinco preguntas breves usando los resultados obtenidos.

En el ejercicio 1 elegís cómo combinar dos búsquedas y qué modelo de embeddings usar. En el 2 escribís las instrucciones que recibe el LLM.

In [ ]:
#@title Preparación del entorno { display-mode: "form" }
# Descarga y carga los modelos. Tarda dos o tres minutos la primera vez.
import time
import unicodedata
import warnings

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as registro_transformers

warnings.filterwarnings("ignore")
registro_transformers.set_verbosity_error()
pd.set_option("display.max_colwidth", 110)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
if DISPOSITIVO == "cpu":
    print("ATENCIÓN: no hay GPU. Activala en Entorno de ejecución > Cambiar tipo de entorno de ejecución")
    print("y volvé a ejecutar esta celda. Sin GPU, el ejercicio 2 tarda demasiado.\n")

inicio = time.time()
MODELOS_EMBEDDINGS = {
    nombre: SentenceTransformer(nombre, device=DISPOSITIVO)
    for nombre in [
        "sentence-transformers/all-MiniLM-L6-v2",
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    ]
}
NOMBRE_LLM = "Qwen/Qwen3-1.7B"
tokenizer_llm = AutoTokenizer.from_pretrained(NOMBRE_LLM, padding_side="left")
llm = AutoModelForCausalLM.from_pretrained(
    NOMBRE_LLM, dtype=torch.float16 if DISPOSITIVO == "cuda" else torch.float32
).to(DISPOSITIVO)

print("Usando:", DISPOSITIVO)
print(f"Modelos cargados en {time.time() - inicio:.0f} segundos")

## Ejercicio 1: buscar el fragmento correcto (15 puntos)

### Objetivo
Mirá los datos, probá distintas combinaciones de búsqueda léxica y semántica con distintos modelos de embeddings, y conseguí recall@3 de al menos 0,85 en validación.

### Recorrido

1. Ejecutá **Datos del ejercicio 1**.
2. Ejecutá la celda **MIRÁ LOS DATOS** y leé las tablas.
3. Leé las decisiones disponibles y las fichas de los modelos. Agregá al menos tres pruebas en **TU TRABAJO**.
4. Compará recall@1 y recall@3.
5. Elegí una configuración final, mirá sus errores y respondé R1 y R2.

In [ ]:
#@title Datos del ejercicio 1 { display-mode: "form" }
FRAGMENTOS = [
    ("Cuenta institucional: contraseña",
     "Para restablecer la contraseña de la cuenta institucional (correo, WebAsignatura y Wi-Fi) entrá a cuentas.ucu.edu.uy y elegí \"Olvidé mi contraseña\". Te llega un código al correo personal registrado, que vence a los 15 minutos. La nueva contraseña debe tener al menos 12 caracteres."),
    ("Cuenta institucional: bloqueo",
     "Después de 5 intentos fallidos de ingreso, la cuenta institucional se bloquea durante 30 minutos. Si pasado ese tiempo sigue bloqueada, hay que ir a la Mesa de Ayuda con la cédula de identidad."),
    ("Wi-Fi",
     "La red inalámbrica para estudiantes es UCU-ESTUDIANTES y se ingresa con el usuario y la contraseña de la cuenta institucional. La red UCU-INVITADOS es solo para visitantes y el acceso dura 24 horas. Cada estudiante puede tener hasta 3 dispositivos conectados a la vez."),
    ("Wi-Fi: error WIFI-403",
     "El mensaje WIFI-403 indica que se superó el límite de dispositivos conectados. Para solucionarlo, desconectá uno de los dispositivos desde cuentas.ucu.edu.uy, en la sección Dispositivos."),
    ("Impresiones: cupo y costos",
     "Cada estudiante tiene 100 páginas de impresión gratuitas por semestre, en blanco y negro. Las páginas extra cuestan $3 en blanco y negro y $10 en color, y el saldo se carga en Tesorería."),
    ("Impresiones: errores de impresora",
     "El código E-217 en la pantalla de la impresora significa que no te queda saldo de páginas. El código E-450 indica papel atascado: no intentes sacarlo, avisá en la Mesa de Ayuda."),
    ("Préstamo de notebooks",
     "La Biblioteca presta notebooks para usar dentro del edificio por un máximo de 4 horas. Se piden en el mostrador presentando la cédula. Si una notebook se devuelve tarde, se suspende el préstamo por una semana."),
    ("Biblioteca: préstamo de libros",
     "Se pueden retirar hasta 3 libros por 7 días, con dos renovaciones desde el catálogo en línea si nadie los reservó. Los libros con etiqueta roja son de reserva y solo se consultan en sala."),
    ("Biblioteca: horarios",
     "La Biblioteca abre de lunes a viernes de 8 a 22 horas y los sábados de 9 a 13 horas. Cierra domingos y feriados. En los períodos de exámenes de julio y diciembre, los sábados abre hasta las 18 horas."),
    ("Salas de estudio grupal",
     "Las salas de estudio grupal, para grupos de 4 a 8 personas, se reservan en reservas.ucu.edu.uy con hasta 48 horas de anticipación y por bloques de 2 horas. Si el grupo no llega en 15 minutos, la reserva se libera."),
    ("Laboratorios de computación",
     "Los laboratorios 201, 202, 203 y 204 se pueden usar para práctica libre cuando no hay clase. El laboratorio 204 tiene computadoras con GPU para proyectos de inteligencia artificial, que se reservan a través del docente del curso. En los laboratorios no se puede comer ni beber."),
    ("Software para estudiantes",
     "Con la cuenta institucional tenés licencia gratuita de Microsoft 365 para instalar en tu computadora. MATLAB y SPSS solo están instalados en las computadoras de los laboratorios."),
    ("Acceso remoto (VPN)",
     "Para usar desde tu casa las bases de datos de artículos científicos de la Biblioteca hay que conectarse a la VPN con el programa FortiClient y la cuenta institucional. Hay guías de instalación para Windows, macOS y Linux."),
    ("Mesa de Ayuda: contacto",
     "La Mesa de Ayuda atiende en la sala 104, de lunes a viernes de 9 a 18 horas, o por correo a ayuda@ucu.edu.uy. Si escribís por correo, incluí tu número de estudiante y una captura del error. El tiempo de respuesta es de hasta 2 días hábiles."),
    ("Credencial de estudiante",
     "La credencial de estudiante se tramita en Secretaría. Si la perdés, la reposición cuesta $250 y demora 5 días hábiles."),
    ("Objetos perdidos",
     "Los objetos encontrados en el edificio se guardan en Portería durante 30 días. Después de ese plazo se donan."),
]
documentos = pd.DataFrame(FRAGMENTOS, columns=["seccion", "texto"])
TEXTOS = (documentos["seccion"] + ": " + documentos["texto"]).tolist()

# (consulta, número del fragmento que la responde o None, dato que tiene que aparecer en la respuesta)
consultas_validacion = pd.DataFrame([
    ("olvidé mi clave, ¿cómo hago para entrar de nuevo?", 0, "cuentas.ucu.edu.uy"),
    ("me equivoqué varias veces al poner la clave y ahora no me deja entrar", 1, "30 minutos"),
    ("¿cómo me conecto a internet desde el celular en la UCU?", 2, "UCU-ESTUDIANTES"),
    ("me aparece WIFI-403, ¿qué significa?", 3, "dispositivos"),
    ("¿cuántas hojas puedo imprimir sin pagar?", 4, "100"),
    ("la impresora muestra E-217", 5, "saldo"),
    ("¿me pueden prestar una compu para trabajar en la universidad?", 6, "4 horas"),
    ("¿por cuánto tiempo me puedo llevar un libro a casa?", 7, "7 días"),
    ("¿a qué hora cierran la biblioteca los sábados?", 8, "13"),
    ("quiero un lugar para estudiar con mi grupo, ¿cómo lo reservo?", 9, "reservas.ucu.edu.uy"),
    ("¿dónde hay máquinas con placa de video para entrenar redes neuronales?", 10, "204"),
    ("¿tengo Word y Excel gratis por ser estudiante?", 11, "Microsoft 365"),
    ("¿cómo leo papers de la biblioteca desde mi casa?", 12, "FortiClient"),
    ("¿en qué horario atiende la Mesa de Ayuda?", 13, "9 a 18"),
    ("perdí el carnet de estudiante, ¿cuánto sale sacar uno nuevo?", 14, "250"),
    ("me olvidé la campera en un salón, ¿dónde la busco?", 15, "Portería"),
    ("¿cuánto cuesta la matrícula del semestre?", None, "No lo sé"),
    ("¿la UCU tiene estacionamiento para autos?", None, "No lo sé"),
    ("¿quién es el director de la Mesa de Ayuda?", None, "No lo sé"),
], columns=["consulta", "fragmento", "esperado"])

consultas_prueba = pd.DataFrame([
    ("no me acuerdo de la contraseña del correo de la UCU", 0, "cuentas.ucu.edu.uy"),
    ("la cuenta me quedó bloqueada y ya pasó media hora", 1, "cédula"),
    ("¿cuántos aparatos puedo tener conectados al wifi?", 2, "3"),
    ("¿cuánto sale imprimir en color?", 4, "10"),
    ("aparece E-450 en la fotocopiadora", 5, "Mesa de Ayuda"),
    ("¿qué pasa si devuelvo tarde la notebook?", 6, "semana"),
    ("¿la biblioteca abre el domingo?", 8, "cierra"),
    ("¿con cuánta anticipación reservo una sala para estudiar en grupo?", 9, "48"),
    ("¿dónde uso MATLAB?", 11, "laboratorio"),
    ("si mando un mail a soporte, ¿en cuánto me responden?", 13, "2 días"),
    ("¿hay cafetería en el edificio?", None, "No lo sé"),
    ("¿puedo pagar las impresiones con tarjeta de crédito?", None, "No lo sé"),
], columns=["consulta", "fragmento", "esperado"])

vectorizador = TfidfVectorizer(strip_accents="unicode")
vectores_tfidf = vectorizador.fit_transform(TEXTOS)
vectores_embeddings = {
    nombre: modelo.encode(TEXTOS, normalize_embeddings=True)
    for nombre, modelo in MODELOS_EMBEDDINGS.items()
}


def palabras_en_comun(consulta, numero_fragmento):
    analizar = vectorizador.build_analyzer()
    return sorted(set(analizar(consulta)) & set(analizar(TEXTOS[numero_fragmento])))


def mostrar_datos_ej1():
    print(len(documentos), "fragmentos de documentación")
    display(documentos.assign(texto=documentos["texto"].str[:90] + "…"))

    print("\nAlgunas consultas de validación y su fragmento correcto:")
    ejemplos = consultas_validacion.iloc[[0, 4, 5, 11]]
    display(pd.DataFrame({
        "consulta": ejemplos["consulta"],
        "fragmento correcto": [documentos["seccion"][int(i)] for i in ejemplos["fragmento"]],
        "palabras en común": [palabras_en_comun(c, int(i)) for c, i in zip(ejemplos["consulta"], ejemplos["fragmento"])],
    }))

    consulta = "¿cuántas hojas puedo imprimir sin pagar?"
    print(f"\nCómo parte cada modelo de embeddings la consulta: {consulta!r}")
    filas = []
    for nombre, modelo in MODELOS_EMBEDDINGS.items():
        tokens = modelo.tokenizer.tokenize(consulta)
        filas.append({"modelo": nombre, "cantidad de tokens": len(tokens), "tokens": " | ".join(tokens)})
    display(pd.DataFrame(filas))


print("Datos del ejercicio 1 preparados:", len(consultas_validacion), "consultas de validación")

In [ ]:
# EJECUTÁ SIN CAMBIOS. Mirá las tablas que aparecen abajo.
mostrar_datos_ej1()

### Decisiones disponibles en el ejercicio 1

El buscador le da a cada fragmento dos puntajes: el de **TF-IDF** (palabras en común con la consulta) y la **similitud coseno** entre los embeddings de la consulta y del fragmento. Los dos se llevan a la escala de 0 a 1 y se combinan así:

```text
puntaje = (1 - peso_semantico) · puntaje TF-IDF + peso_semantico · similitud de embeddings
```

Cada prueba es una fila `(peso_semantico, modelo)`:

- **peso_semantico**: un número entre 0 y 1. Pensá qué búsqueda obtenés con 0, con 1 y con valores intermedios.
- **modelo**: el identificador del modelo de embeddings en Hugging Face, entre comillas. Ya están descargados estos dos. Leé sus fichas (*model cards*) para saber con qué datos fueron entrenados:

| Modelo | Ficha |
|---|---|
| `sentence-transformers/all-MiniLM-L6-v2` | [huggingface.co/sentence-transformers/all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) |
| `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` | [huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) |

Si `peso_semantico` es 0, el modelo no se usa y podés escribir `None`.

Las métricas se calculan con las 16 consultas de validación que tienen respuesta en la documentación:

- **recall@1**: proporción de consultas en las que el fragmento correcto quedó **primero**.
- **recall@3**: proporción de consultas en las que el fragmento correcto quedó **entre los 3 primeros**.

In [ ]:
#@title Herramientas preparadas del ejercicio 1 { display-mode: "form" }
OBJETIVO_RECALL_EJ1 = 0.85


def escalar_0_1(puntajes):
    return (puntajes - puntajes.min()) / (puntajes.max() - puntajes.min() + 1e-9)


def nombre_modelo(modelo):
    if modelo in MODELOS_EMBEDDINGS:
        return modelo
    if isinstance(modelo, str) and "sentence-transformers/" + modelo in MODELOS_EMBEDDINGS:
        return "sentence-transformers/" + modelo
    raise ValueError("Modelo no disponible. Usá uno de estos (entre comillas): " + ", ".join(MODELOS_EMBEDDINGS))


def puntajes_busqueda(consulta, peso_semantico, modelo):
    if not isinstance(peso_semantico, (int, float)) or not 0 <= peso_semantico <= 1:
        raise ValueError("peso_semantico tiene que ser un número entre 0 y 1")
    lexica = (vectorizador.transform([consulta]) @ vectores_tfidf.T).toarray()[0]
    if peso_semantico == 0:
        return lexica
    modelo = nombre_modelo(modelo)
    vector = MODELOS_EMBEDDINGS[modelo].encode(consulta, normalize_embeddings=True)
    semantica = vectores_embeddings[modelo] @ vector
    if peso_semantico == 1:
        return semantica
    return (1 - peso_semantico) * escalar_0_1(lexica) + peso_semantico * escalar_0_1(semantica)


def buscar(consulta, peso_semantico, modelo, k=3):
    return list(np.argsort(-puntajes_busqueda(consulta, peso_semantico, modelo))[:k])


def evaluar_busqueda(configuracion, consultas):
    peso_semantico, modelo = configuracion
    con_respuesta = consultas.dropna(subset=["fragmento"])
    encontrados = [buscar(c, peso_semantico, modelo) for c in con_respuesta["consulta"]]
    correctos = con_respuesta["fragmento"].astype(int).tolist()
    return {
        "peso_semantico": peso_semantico,
        "modelo": "no se usa" if peso_semantico == 0 else nombre_modelo(modelo),
        "recall@1": round(np.mean([c in e[:1] for c, e in zip(correctos, encontrados)]), 2),
        "recall@3": round(np.mean([c in e[:3] for c, e in zip(correctos, encontrados)]), 2),
        "detalle": pd.DataFrame({
            "consulta": con_respuesta["consulta"].tolist(),
            "fragmento correcto": [documentos["seccion"][c] for c in correctos],
            "puesto": [e.index(c) + 1 if c in e else "fuera del top 3" for c, e in zip(correctos, encontrados)],
            "primer resultado": [documentos["seccion"][e[0]] for e in encontrados],
        }),
    }


def ejecutar_trabajo_ej1(pruebas, configuracion_final):
    visibles = ["peso_semantico", "modelo", "recall@1", "recall@3"]
    display(pd.DataFrame([evaluar_busqueda(p, consultas_validacion) for p in pruebas])[visibles])
    resultado = evaluar_busqueda(configuracion_final, consultas_validacion)
    print("Configuración final: peso_semantico =", resultado["peso_semantico"], "/ modelo:", resultado["modelo"])
    print("recall@1 validación:", resultado["recall@1"])
    print("recall@3 validación:", resultado["recall@3"])
    print("Objetivo: recall@3 de al menos", OBJETIVO_RECALL_EJ1)
    print("\nConsultas en las que el fragmento correcto NO quedó primero:")
    detalle = resultado["detalle"]
    display(detalle[detalle["puesto"] != 1])
    return resultado

In [ ]:
# TU TRABAJO · EJERCICIO 1 (15 puntos)
# EDITÁ SOLO las listas PRUEBAS_EJ1 y CONFIGURACION_FINAL_EJ1.
# Cada fila es (peso_semantico, modelo).
PRUEBAS_EJ1 = [
    (0, None),
    # Agregá al menos tres pruebas.
]

# Elegí una configuración que hayas probado y que alcance el objetivo.
CONFIGURACION_FINAL_EJ1 = (0, None)
resultado_final_ej1 = ejecutar_trabajo_ej1(PRUEBAS_EJ1, CONFIGURACION_FINAL_EJ1)
RECALL_VALIDACION_EJ1 = resultado_final_ej1["recall@3"]

## Ejercicio 2: responder con RAG (15 puntos)

### Objetivo
Mirá qué responde el LLM sin contexto, escribí instrucciones para el LLM, compará valores de k y conseguí al menos 55% de respuestas correctas y al menos 2 de 3 abstenciones.

El LLM es [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B): un modelo chico de pesos abiertos (1.700 millones de parámetros). Genera con temperatura 0. Para buscar los fragmentos se usa **tu configuración final del ejercicio 1** (`CONFIGURACION_FINAL_EJ1`). Si la cambiás, volvé a ejecutar la celda **TU TRABAJO** del ejercicio 2.

### Recorrido

1. Ejecutá **Herramientas del ejercicio 2**.
2. Ejecutá la celda **MIRÁ EL LLM**: leé la respuesta sin contexto y cómo se arma el prompt.
3. En **TU TRABAJO**, escribí al menos dos instrucciones propias y compará al menos cuatro configuraciones. Cada una tarda unos segundos con GPU.
4. Elegí la configuración final y mirá sus respuestas una por una.
5. Respondé R3 y R4 usando la tabla.

### Decisiones disponibles

Cada prueba es una fila `(instruccion, k)`:

- **instruccion**: el texto del mensaje `system`, entre comillas. Es lo único del prompt que escribís vos. El mensaje `user` se arma solo, con los fragmentos encontrados y la pregunta (lo ves en la celda **MIRÁ EL LLM**).
- **k**: cuántos fragmentos se pegan en el prompt: `1`, `3` o `5`. Con `0` no se pega ningún fragmento y el mensaje `user` es solo la pregunta (sin RAG).

### Qué mide la tabla

- **recall@k**: proporción de las 16 consultas con respuesta en las que el fragmento correcto llegó al prompt.
- **correctas**: proporción de esas 16 en las que la respuesta contiene el dato esperado.
- **abstenciones**: de las 3 consultas **sin** respuesta en la documentación, en cuántas la respuesta contiene la frase **"No lo sé"**.
- **tokens_prompt**: largo promedio del prompt, en tokens. Más tokens es más costo y más demora.

In [ ]:
#@title Herramientas del ejercicio 2 { display-mode: "form" }
OBJETIVO_CORRECTAS_EJ2 = 0.55
OBJETIVO_ABSTENCIONES_EJ2 = 2
_respuestas_guardadas = {}


def sin_tildes(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto.lower()) if unicodedata.category(c) != "Mn")


def armar_prompt(consulta, instruccion, k):
    if not isinstance(instruccion, str) or not instruccion.strip():
        raise ValueError("La instrucción tiene que ser un texto entre comillas")
    if k not in (0, 1, 3, 5):
        raise ValueError(f"k tiene que ser 0, 1, 3 o 5, no {k!r}")
    if k == 0:
        return [
            {"role": "system", "content": instruccion},
            {"role": "user", "content": consulta},
        ], []
    encontrados = buscar(consulta, *CONFIGURACION_FINAL_EJ1, k=k)
    contexto = "\n".join(f"[{n}] {TEXTOS[i]}" for n, i in enumerate(encontrados, start=1))
    return [
        {"role": "system", "content": instruccion},
        {"role": "user", "content": f"Contexto:\n{contexto}\n\nPregunta: {consulta}"},
    ], encontrados


def generar_lote(lista_mensajes, max_tokens=60):
    # Genera todas las respuestas juntas (en lote), con temperatura 0.
    textos = [
        tokenizer_llm.apply_chat_template(
            mensajes, add_generation_prompt=True, tokenize=False, enable_thinking=False
        )
        for mensajes in lista_mensajes
    ]
    ids = tokenizer_llm(textos, return_tensors="pt", padding=True).to(DISPOSITIVO)
    with torch.no_grad():
        salida = llm.generate(
            **ids, do_sample=False, max_new_tokens=max_tokens,
            pad_token_id=tokenizer_llm.pad_token_id,
        )
    respuestas = tokenizer_llm.batch_decode(salida[:, ids["input_ids"].shape[1]:], skip_special_tokens=True)
    tokens = [len(tokenizer_llm(t)["input_ids"]) for t in textos]
    return [r.strip() for r in respuestas], tokens


def responder_todas(consultas, instruccion, k):
    claves = [(c, instruccion, k, CONFIGURACION_FINAL_EJ1) for c in consultas]
    faltan = [c for c, clave in zip(consultas, claves) if clave not in _respuestas_guardadas]
    if faltan:
        armados = [armar_prompt(c, instruccion, k) for c in faltan]
        respuestas, tokens = generar_lote([mensajes for mensajes, _ in armados])
        for c, (_, encontrados), r, t in zip(faltan, armados, respuestas, tokens):
            _respuestas_guardadas[(c, instruccion, k, CONFIGURACION_FINAL_EJ1)] = (r, encontrados, t)
    return [_respuestas_guardadas[clave] for clave in claves]


def es_correcta(respuesta, esperado):
    return sin_tildes(esperado) in sin_tildes(respuesta)


def evaluar_rag(configuracion, consultas):
    instruccion, k = configuracion
    inicio = time.time()
    filas = []
    salidas = responder_todas(consultas["consulta"].tolist(), instruccion, k)
    for (_, caso), (respuesta, encontrados, tokens) in zip(consultas.iterrows(), salidas):
        tiene_respuesta = not pd.isna(caso["fragmento"])
        filas.append({
            "consulta": caso["consulta"],
            "esperado": caso["esperado"],
            "recuperado": (int(caso["fragmento"]) in encontrados) if tiene_respuesta and encontrados else None,
            "correcta": es_correcta(respuesta, caso["esperado"]),
            "con_respuesta": tiene_respuesta,
            "fragmentos": [documentos["seccion"][i] for i in encontrados],
            "respuesta": respuesta,
            "tokens": tokens,
        })
    detalle = pd.DataFrame(filas)
    con = detalle[detalle["con_respuesta"]]
    sin = detalle[~detalle["con_respuesta"]]
    return {
        "instruccion": instruccion,
        "k": k,
        "recall@k": "no aplica" if k == 0 else round(con["recuperado"].mean(), 2),
        "correctas": round(con["correcta"].mean(), 2),
        "abstenciones": f"{int(sin['correcta'].sum())} de {len(sin)}",
        "abstenciones_n": int(sin["correcta"].sum()),
        "tokens_prompt": round(detalle["tokens"].mean()),
        "segundos": round(time.time() - inicio),
        "detalle": detalle,
    }


def mostrar_llm_ej2():
    instruccion = "Sos un asistente."
    consulta = "¿cuántas hojas puedo imprimir sin pagar?"
    respuesta = generar_lote([armar_prompt(consulta, instruccion, 0)[0]])[0][0]
    print("SIN CONTEXTO (k=0)")
    print("Pregunta: ", consulta)
    print("Respuesta:", respuesta)
    print("Según la documentación: 100 páginas por semestre.\n")

    mensajes, _ = armar_prompt(consulta, instruccion, 3)
    print("ASÍ SE ARMA EL PROMPT CON k=3")
    print("--- system (lo escribís vos) ---")
    print(mensajes[0]["content"])
    print("--- user (se arma solo) ---")
    print(mensajes[1]["content"])


def ejecutar_trabajo_ej2(configuraciones, configuracion_final):
    resultados = [evaluar_rag(c, consultas_validacion) for c in configuraciones]
    letras = {}
    for r in resultados:
        letras.setdefault(r["instruccion"], "ABCDEFGHIJKLMNOPQRSTUVWXYZ"[len(letras) % 26])
    print("Instrucciones probadas:")
    for texto, letra in letras.items():
        print(f"  {letra}: {texto}")
    tabla = pd.DataFrame(resultados)
    tabla["instruccion"] = tabla["instruccion"].map(letras)
    visibles = ["instruccion", "k", "recall@k", "correctas", "abstenciones", "tokens_prompt", "segundos"]
    display(tabla[visibles])
    resultado = evaluar_rag(configuracion_final, consultas_validacion)
    print("Configuración final: instrucción", letras.get(resultado["instruccion"], "(no probada en la lista)"), "con k =", resultado["k"])
    print("Correctas en validación:", resultado["correctas"])
    print("Abstenciones en validación:", resultado["abstenciones"])
    print(f"Objetivo: correctas de al menos {OBJETIVO_CORRECTAS_EJ2} y al menos {OBJETIVO_ABSTENCIONES_EJ2} de 3 abstenciones")
    print("\nRespuestas de la configuración final:")
    display(resultado["detalle"][["consulta", "esperado", "recuperado", "correcta", "respuesta"]])
    return resultado


print("Herramientas del ejercicio 2 preparadas")

In [ ]:
# EJECUTÁ SIN CAMBIOS. Leé la respuesta sin contexto y cómo se arma el prompt.
mostrar_llm_ej2()

In [ ]:
# TU TRABAJO · EJERCICIO 2 (15 puntos)
# EDITÁ SOLO las instrucciones, CONFIGURACIONES_EJ2 y CONFIGURACION_FINAL_EJ2.
INSTRUCCION_BASE = "Sos un asistente."

# Escribí tus instrucciones. Podés partir de INSTRUCCION_BASE y agregarle texto.
MI_INSTRUCCION_1 = INSTRUCCION_BASE + " "
MI_INSTRUCCION_2 = INSTRUCCION_BASE + " "

# Cada fila es (instruccion, k).
CONFIGURACIONES_EJ2 = [
    (INSTRUCCION_BASE, 0),
    (INSTRUCCION_BASE, 1),
    # Agregá filas con tus instrucciones y distintos valores de k.
]

# Elegí una configuración que hayas probado y que alcance el objetivo.
CONFIGURACION_FINAL_EJ2 = (INSTRUCCION_BASE, 1)
resultado_final_ej2 = ejecutar_trabajo_ej2(CONFIGURACIONES_EJ2, CONFIGURACION_FINAL_EJ2)

## Comprobación final

Cuando ambos objetivos de validación estén cumplidos, ejecutá la comprobación una sola vez. No modifiques esa celda.

In [ ]:
#@title Comprobación final { display-mode: "form" }
# EJECUTÁ SIN CAMBIOS. Si falta mejorar algo, la celda lo informa sin detener el cuaderno.
objetivos_cumplidos = (
    RECALL_VALIDACION_EJ1 >= OBJETIVO_RECALL_EJ1
    and resultado_final_ej2["correctas"] >= OBJETIVO_CORRECTAS_EJ2
    and resultado_final_ej2["abstenciones_n"] >= OBJETIVO_ABSTENCIONES_EJ2
)

if not objetivos_cumplidos:
    print("Comprobación final pendiente: revisá las elecciones de los ejercicios 1 y 2.")
else:
    prueba_ej1 = evaluar_busqueda(CONFIGURACION_FINAL_EJ1, consultas_prueba)
    prueba_ej2 = evaluar_rag(CONFIGURACION_FINAL_EJ2, consultas_prueba)
    print("Ejercicio 1 · recall@1 de prueba (test):", prueba_ej1["recall@1"])
    print("Ejercicio 1 · recall@3 de prueba (test):", prueba_ej1["recall@3"])
    print("Ejercicio 2 · correctas de prueba (test):", prueba_ej2["correctas"])
    print("Ejercicio 2 · abstenciones de prueba (test):", prueba_ej2["abstenciones"])
    print("Comprobación final: OK")
    display(prueba_ej2["detalle"][["consulta", "esperado", "correcta", "respuesta"]])

## Respuestas cortas (20 puntos)

Hacé doble clic en esta celda y respondé usando tus resultados.

### R1. Qué viste en los datos

En la celda **MIRÁ LOS DATOS** del ejercicio 1, elegí una consulta que **no** comparte palabras con su fragmento correcto y otra que sí. ¿Qué tipo de búsqueda debería encontrar cada una, y por qué? Comentá además una diferencia en cómo los dos modelos parten la consulta en tokens. **Máximo: 10 líneas.**

**Respuesta:** …

### R2. Qué búsqueda elegiste

¿Qué peso_semantico y qué modelo elegiste? Explicá qué búsqueda obtenés con ese peso y compará al menos dos pruebas con recall@1 y recall@3. ¿Por qué uno de los modelos de embeddings funciona peor con estas consultas? Usá su ficha y la tokenización de **MIRÁ LOS DATOS**. Mirá la tabla de errores de tu configuración final: elegí una consulta mal ubicada y proponé una explicación. **Máximo: 10 líneas.**

**Respuesta:** …

### R3. Sin RAG y con RAG

¿Qué pasó con la respuesta del LLM sin contexto en la celda **MIRÁ EL LLM** y con k=0 en la tabla? Explicá por qué un LLM responde así. Copiá tu instrucción final y explicá qué agregaste respecto de `INSTRUCCION_BASE` y para qué sirve cada parte. Compará con otra instrucción que hayas probado usando correctas y abstenciones. **Máximo: 10 líneas.**

**Respuesta:** …

### R4. Qué valor de k elegiste

Explicá qué k elegiste usando recall@k, correctas y tokens_prompt. En las respuestas de tu configuración final, elegí una marcada como incorrecta (o, si no hay, una dudosa) y decí si falló la **recuperación** o la **generación**, y cómo lo sabés. ¿La regla "contiene el dato esperado" la evaluó bien? **Máximo: 10 líneas.**

**Respuesta:** …

### R5. Para qué se reserva la prueba final

¿Por qué usamos las consultas de validación para elegir la búsqueda, la instrucción y k, y abrimos las consultas de prueba (test) una sola vez al final? ¿Qué pasaría si probáramos muchos prompts mirando directamente la prueba? **Máximo: 10 líneas.**

**Respuesta:** …